# Efficient-SAM3-Finetuning — GPU Test Runner

One-click runner for GPU-gated tests (currently: `spec/peft-qlora`).

**Prereqs:**
1. Runtime → Change runtime type → **T4 GPU** (or better).
2. In Colab Secrets (left sidebar 🔑), add:
   - `HF_TOKEN` — Hugging Face access token with read access to gated `facebook/sam3.1`.
   - `GH_TOKEN` — GitHub fine-grained personal access token with **Contents: Read** on this repo. **Required only if the repo is private**; can be omitted for public repos.
   - `BRANCH` *(optional)* — feature branch to test (e.g. `worktree-fix+colab-remaining-failures`). Used only if the notebook cannot detect the branch from the Colab URL.
3. Runtime → Run all. The notebook auto-detects `BRANCH` (URL → notebook metadata → `BRANCH` secret → `main`). Override by editing `BRANCH = "..."` in the next cell.

In [ ]:
# Cell 0: Config. Auto-detects the working branch so you don't edit the
# notebook on every test cycle. Resolution order (first hit wins):
#   1. BRANCH literal set below (manual override).
#   2. URL parse from the Colab tab (works when opened via "Open in Colab"
#      on a GitHub branch — uses document.referrer because cell-output JS
#      runs in a cross-origin iframe that cannot read top.location).
#   3. metadata.colab.provenance from google.colab._message — populated
#      by Colab when the notebook was loaded from GitHub.
#   4. BRANCH Colab Secret (left sidebar key). Set this once on a branch
#      you iterate on and you never touch the notebook again.
#   5. "main".
import re
import urllib.parse

REPO = "NguyenJus/Efficient-SAM3-Finetuning"
BRANCH = None  # set to a string literal to force a branch

_GITHUB_URL_RE = re.compile(r"/github/[^/]+/[^/]+/blob/([^/?#]+)/", re.IGNORECASE)


def _from_url() -> str | None:
    try:
        from google.colab import output  # type: ignore
    except Exception:
        return None
    try:
        url = output.eval_js(
            "(() => { try { return document.referrer || ''; } catch(e) { return ''; } })()",
            timeout_sec=5,
        )
    except Exception:
        return None
    if not url:
        return None
    m = _GITHUB_URL_RE.search(url)
    return urllib.parse.unquote(m.group(1)) if m else None


def _from_ipynb_metadata() -> str | None:
    try:
        from google.colab import _message  # type: ignore

        meta = _message.blocking_request("get_ipynb", timeout_sec=5)["ipynb"]["metadata"]
    except Exception:
        return None
    for p in meta.get("colab", {}).get("provenance", []) or []:
        for v in p.values() if isinstance(p, dict) else []:
            if isinstance(v, str):
                m = _GITHUB_URL_RE.search(v)
                if m:
                    return urllib.parse.unquote(m.group(1))
    return None


def _from_secret() -> str | None:
    try:
        from google.colab import userdata  # type: ignore

        v = userdata.get("BRANCH")
    except Exception:
        return None
    return v or None


if BRANCH is None:
    for label, fn in (
        ("url", _from_url),
        ("metadata", _from_ipynb_metadata),
        ("secret", _from_secret),
    ):
        candidate = fn()
        if candidate:
            BRANCH, _source = candidate, label
            break
    else:
        BRANCH, _source = "main", "fallback"
else:
    _source = "literal"

print(f"REPO   = {REPO}")
print(f"BRANCH = {BRANCH}  (source: {_source})")

In [ ]:
# Cell 1: Clone + checkout. Done BEFORE any torch/numpy import so the pip
# install in Cell 2 can downgrade numpy without Colab forcing a runtime restart.
import os
import subprocess

try:
    from google.colab import userdata  # type: ignore

    gh_token = userdata.get("GH_TOKEN")
except Exception:
    gh_token = None

clone_url = (
    f"https://{gh_token}@github.com/{REPO}.git" if gh_token else f"https://github.com/{REPO}.git"
)

if not os.path.isdir("Efficient-SAM3-Finetuning"):
    subprocess.run(["git", "clone", clone_url], check=True)
subprocess.run(["git", "-C", "Efficient-SAM3-Finetuning", "fetch", "--all"], check=True)
subprocess.run(["git", "-C", "Efficient-SAM3-Finetuning", "checkout", BRANCH], check=True)
os.chdir("Efficient-SAM3-Finetuning")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
# Cell 2: Install runtime + GPU + test deps. Runs BEFORE Cell 3's torch import
# so numpy 1.26.4 lands cleanly — no "You must restart the runtime" warning.
#
# Why all these pins on ONE pip-install line:
#
# Colab ships numpy 2.x, scipy 1.16.x, and transformers 5.0.0; sam3 pins
# `numpy>=1.26,<2`. If we let pip's resolver loose with just `numpy>=1.26`
# from our project, it tries numpy 2.x first (latest matching), hits sam3's
# `<2` cap, and cascades: it backtracks through every transformers version
# (5.8.1 → 5.0.0), then every scipy (1.17.1 → 1.11.1-sdist), and the scipy
# sdist fails to build from source → `metadata-generation-failed`.
#
# Splitting these into a separate pre-install does NOT work: pip re-resolves
# from scratch on the next `pip install` and re-considers upgrading numpy
# even though 1.26.4 is on disk. The pins must be on the SAME install
# command so pip honors them during the single resolution pass.
#
# Re-evaluate these pins if sam3 ever relaxes its numpy bound. The
# transformers pin is the version Colab preinstalls; bumping it is fine as
# long as the bumped version is also in the kernel's site-packages.
#
# torchao>=0.16.0 is pinned because peft 0.19+ lazily checks the installed
# torchao version on every LoRA-eligible nn.Linear dispatch
# (peft.tuners.lora.torchao calls peft.import_utils.is_torchao_available,
# which RAISES ImportError if torchao is installed AND `< 0.16.0`). Colab
# base images preinstall torchao 0.10.0, so apply_lora and apply_qlora hit
# this ImportError before any of our code runs. We do NOT use torchao
# ourselves (QLoRA uses bitsandbytes); the pin only upgrades the
# preinstalled package past peft's gate. Drop this pin once Colab ships
# torchao >= 0.16.0 by default.
%pip install -e ".[qlora,dev,tensorboard]" \
    "numpy==1.26.4" "scipy==1.13.1" "transformers==5.0.0" \
    "huggingface_hub>=1.15" \
    "torchao>=0.16.0"
!python -c "import esam3; print('esam3 OK:', esam3.__file__)"

In [ ]:
# Cell 3: Environment guard. Runs AFTER the install so torch imports a numpy
# that already matches sam3's pin — fails fast if anything drifted.
import sys

import numpy
import torch

assert "google.colab" in sys.modules, "This notebook is intended for Google Colab."
assert numpy.__version__.startswith("1.26."), (
    f"numpy {numpy.__version__} != 1.26.x. Cell 2 install drifted; "
    "Runtime → Restart runtime, then Run all."
)
assert torch.cuda.is_available(), (
    "No CUDA device. Runtime → Change runtime type → T4 GPU or better."
)
cc = torch.cuda.get_device_capability()
assert cc >= (7, 5), (
    f"CUDA compute capability {cc} is < 7.5. bitsandbytes 4-bit requires Turing or later."
)
print(f"GPU OK: {torch.cuda.get_device_name(0)} (CC {cc[0]}.{cc[1]})")
print(f"numpy: {numpy.__version__}")

In [ ]:
# Cell 4: HF auth (token in Colab Secrets).
import os

from google.colab import userdata

token = userdata.get("HF_TOKEN")
assert token, "HF_TOKEN missing from Colab Secrets. See the prereqs cell."
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token  # huggingface-cli reads this name too

In [ ]:
# Cell 5: Download the SAM 3.1 checkpoint (gated; HF_TOKEN required).
# `huggingface-cli` is deprecated; the new CLI is `hf` (huggingface_hub >= 1.13).
!hf download facebook/sam3.1 --local-dir models/sam3.1

In [ ]:
# Cell 6: Run the gated tests.
!bash scripts/run_gpu_tests.sh

## Cell 7: Summary

Scroll to the bottom of cell 6's output and read pytest's final summary line
(e.g. `========= 4 passed in 87.3s =========`). That line is the pass/fail signal.